# 1.2 Join HDB block polygons to HDB Property Information

The demand side of the study is children living in HDB blocks, so every block polygon needs its
attributes: floors, dwelling units, flat-type mix and use flags (residential, carpark, market ...).
The two public datasets share no key. The polygons (HDB Existing Building) identify a block by
`BLK_NO` + `ST_COD`, an LTA road code; the attribute table (HDB Property Information) uses
`blk_no` + an abbreviated street name ("BEDOK STH AVE 1"). This notebook decodes the road code with
LTA's lookup table, rewrites full-word suffixes into HDB's abbreviations (AVENUE → AVE, SOUTH → STH,
ROAD → RD ...) and joins on block + street. Without the abbreviation step only about 10% of blocks
match; with it 98.8% match on the first pass, and two fallback passes plus a manual review of four
blocks bring the final figure to 99.1%. Original script: `scripts/join_hdb_nationwide.py` (Minh).

**Inputs**
- `data/HDBExistingBuilding_2026.geojson` — HDB Existing Building, data.gov.sg
  `d_16b157c52ed637edd6ba1232e026258d`, version 2026-06-21 (13,436 polygons).
- `data/HDBPropertyInformation_2026.csv` — HDB Property Information, data.gov.sg
  `d_17f5382f26140b1fdae0ba2ef6239d2f`, version 2026-10-08.
- `data/LTARoadNameCode_2019.xlsx` — LTA DataMall road-code → road-name lookup, sheet `GISDomainData`.

**Outputs**
- `output/hdb_blocks.gpkg`, layer `hdb_blocks` — all 13,436 blocks, EPSG:3414, with the CSV
  attributes plus `footprint_m2`, `gfa_m2` (footprint × floors), `is_residential`, `match_status`,
  `review_flag`; read by 1_03, 1_04, 1_08.
- `output/hdb_join_review.csv` — the 120 blocks still without a Property Information row after the
  fallback passes (117 excluded, 3 included with imputed units), for the record.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import pyogrio

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "bt4015").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from bt4015.paths import DATA, OUT, FIG, LARGE

HDB_GEOJSON = DATA / "HDBExistingBuilding_2026.geojson"
HDB_PROPERTY_CSV = DATA / "HDBPropertyInformation_2026.csv"
ROAD_CODE_XLSX = DATA / "LTARoadNameCode_2019.xlsx"

OUT_GPKG = OUT / "hdb_blocks.gpkg"
OUT_LAYER = "hdb_blocks"
REVIEW_CSV = OUT / "hdb_join_review.csv"

CRS_OUT = "EPSG:3414"      # SVY21 / Singapore TM, metres
IMPUTE_NEIGHBOURS = 3      # same-street residential blocks used to impute a manually included block

# Columns carried over from the CSV (beyond the join key).
PROPERTY_COLS = [
    "year_completed", "max_floor_lvl", "bldg_contract_town",
    "residential", "commercial", "market_hawker", "miscellaneous",
    "multistorey_carpark", "precinct_pavilion",
    "total_dwelling_units",
    "1room_sold", "2room_sold", "3room_sold", "4room_sold", "5room_sold",
    "exec_sold", "multigen_sold", "studio_apartment_sold",
    "1room_rental", "2room_rental", "3room_rental", "other_room_rental",
]
UNIT_COLS = [c for c in PROPERTY_COLS if c.endswith(("_sold", "_rental"))]

## 1. Load the block polygons and decode the road code

`ST_COD` is an LTA road code (e.g. `BEDSA1` for Bedok South Avenue 1). The 2019 LTA table decodes
all but two codes, `HOS21Q` and `HOS22Q`, which are Hougang streets built after the table was
published; section 4 recovers those blocks another way.

In [2]:
# 1. HDB Existing Building: one polygon per block, identified by BLK_NO + ST_COD.
hdb = pyogrio.read_dataframe(HDB_GEOJSON)
print(f"HDB block polygons: {len(hdb):,} ({hdb.crs.to_string()})")

# 2. LTA road-code table: header is on row 3 of the sheet, hence skiprows=2.
road = pd.read_excel(ROAD_CODE_XLSX, sheet_name="GISDomainData", skiprows=2)
road.columns = ["S_No", "Road_Code", "Road_Name"]
road = road.dropna(subset=["Road_Code"])
road_lookup = dict(zip(road["Road_Code"], road["Road_Name"]))

# 3. Decode; codes newer than the 2019 table stay null and are handled by the postal-sector pass.
hdb["decoded_street"] = hdb["ST_COD"].map(road_lookup)
undecoded = hdb["decoded_street"].isna()
print(f"Road codes in the LTA table: {len(road_lookup):,}")
print(f"Blocks with an undecoded ST_COD: {int(undecoded.sum())} "
      f"(codes: {', '.join(sorted(hdb.loc[undecoded, 'ST_COD'].unique()))})")

HDB block polygons: 13,436 (EPSG:4326)


Road codes in the LTA table: 5,620
Blocks with an undecoded ST_COD: 19 (codes: HOS21Q, HOS22Q)


## 2. Rewrite street names into HDB's abbreviations

LTA spells suffixes in full (AVENUE, CRESCENT, SOUTH); HDB Property Information abbreviates them
(AVE, CRES, STH). `ABBR` is applied word by word to every decoded street name. Two abbreviations HDB
uses only in some names (`MARKET → MKT`, `INDUSTRIAL → IND`) are kept apart in `EXTRA_ABBR` and tried
only as a fallback, so they do not break the names that keep the full word.

In [3]:
# LTA spells suffixes in full; HDB Property Information abbreviates them.
ABBR = {
    "ROAD": "RD", "STREET": "ST", "AVENUE": "AVE", "DRIVE": "DR", "CRESCENT": "CRES",
    "JALAN": "JLN", "LORONG": "LOR", "NORTH": "NTH", "SOUTH": "STH", "CENTRAL": "CTRL",
    "PLACE": "PL", "TERRACE": "TER", "GARDENS": "GDNS", "GARDEN": "GDN", "PARK": "PK",
    "CLOSE": "CL", "HEIGHTS": "HTS", "GROVE": "GR", "BOULEVARD": "BLVD", "HIGHWAY": "HWY",
    "UPPER": "UPP", "BUKIT": "BT", "KAMPONG": "KG", "TANJONG": "TG", "TAMAN": "TMN",
    "GREEN": "GRN", "COMMONWEALTH": "C'WEALTH", "SQUARE": "SQ",
}
# Abbreviations HDB uses in only some street names, tried as a fallback.
EXTRA_ABBR = {"MARKET": "MKT", "INDUSTRIAL": "IND"}


def normalize_street(name):
    """Upper-case, strip and abbreviate each word of a street name with ABBR."""
    if not isinstance(name, str):
        return None
    return " ".join(ABBR.get(w, w) for w in name.strip().upper().split())


def normalize_blk(blk):
    """Block numbers as upper-case strings ('12A'), null stays null."""
    return str(blk).strip().upper() if pd.notna(blk) else blk


# 1. The join key is "<block>_<abbreviated street>"; undecoded streets give "<block>_" and never match.
hdb["norm_street"] = hdb["decoded_street"].map(normalize_street)
hdb["blk_norm"] = hdb["BLK_NO"].map(normalize_blk)
hdb["join_key"] = hdb["blk_norm"] + "_" + hdb["norm_street"].fillna("")
print(hdb[["BLK_NO", "ST_COD", "decoded_street", "norm_street", "join_key"]].head(8).to_string(index=False))

BLK_NO ST_COD       decoded_street      norm_street             join_key
   277 TOG02Q        TOH GUAN ROAD      TOH GUAN RD      277_TOH GUAN RD
   625 SEN12T           SENJA ROAD         SENJA RD         625_SENJA RD
   296 COC08K COMPASSVALE CRESCENT COMPASSVALE CRES 296_COMPASSVALE CRES
   232 SUL11B          SUMANG LANE      SUMANG LANE      232_SUMANG LANE
    41 HOD00F        HOLLAND DRIVE       HOLLAND DR        41_HOLLAND DR
   509 JEL03J        JELAPANG ROAD      JELAPANG RD      509_JELAPANG RD
   183 BOA01Y      BOON LAY AVENUE     BOON LAY AVE     183_BOON LAY AVE
   810 WOS05J  WOODLANDS STREET 81  WOODLANDS ST 81  810_WOODLANDS ST 81


## 3. Build the same key on HDB Property Information

The CSV already uses the abbreviated spelling, so its key is just block + upper-cased street. The
comparison at the end is the evidence for section 2: with LTA's full-word spelling only about one
block in ten finds its row.

In [4]:
# 1. Property Information: one row per block, keyed by blk_no + street (abbreviated spelling).
prop = pd.read_csv(HDB_PROPERTY_CSV)
prop["blk_norm"] = prop["blk_no"].map(normalize_blk)
prop["street_norm"] = prop["street"].str.strip().str.upper()
prop["join_key"] = prop["blk_norm"] + "_" + prop["street_norm"]

# 2. A duplicated key would double rows in the merge; keep the first occurrence if any.
dup = prop["join_key"].duplicated().sum()
if dup:
    print(f"WARNING: {dup} duplicate keys in property CSV, keeping first")
    prop = prop.drop_duplicates("join_key")
prop_keys = set(prop["join_key"])
print(f"Property Information rows: {len(prop):,}; residential rows: {(prop['residential'] == 'Y').sum():,}")

# 3. Why the abbreviation table matters: share of blocks whose key exists in the CSV.
raw_key = hdb["blk_norm"] + "_" + hdb["decoded_street"].str.strip().str.upper()
print(f"Blocks whose key is in Property Information: with LTA's full spelling {raw_key.isin(prop_keys).mean():.1%}, "
      f"with HDB abbreviations {hdb['join_key'].isin(prop_keys).mean():.1%}")

Property Information rows: 13,392; residential rows: 10,817
Blocks whose key is in Property Information: with LTA's full spelling 10.2%, with HDB abbreviations 98.8%


## 4. Resolve the keys with two fallback passes

Pass 0 is the direct key match (`exact` when the LTA spelling already matched, `normalized` when the
abbreviations were needed). Two fallbacks follow for the blocks still unmatched:

1. `alt_spelling` — try the unabbreviated LTA name (HDB keeps some names in full, e.g. "TAMAN HO
   SWEE", "BUANGKOK GREEN") or the extra abbreviations ("NEW MKT RD"). A candidate is accepted only
   if no other block has already used it.
2. `postal_sector` — for road codes newer than the 2019 LTA table there is no street name at all.
   The block's postal sector (first two digits of the postal code) is mapped to the HDB contract
   towns of already-matched blocks in that sector, and the block number is looked for among the
   unused Property Information rows of those towns. Accepted only when exactly one row qualifies.

Result: 98.8% matched on pass 0, 99.1% after the fallbacks (36 more blocks).

In [5]:
def resolve_keys(hdb, prop):
    """Return (join_key, match_status) per block, with two fallback passes.

    1. alt_spelling: the unabbreviated LTA name (HDB keeps e.g. "TAMAN HO SWEE",
       "BUANGKOK GREEN") or EXTRA_ABBR on top of ABBR ("NEW MKT RD").
    2. postal_sector: for road codes newer than the 2019 LTA table, match the block
       number among property rows in the same contract town, the town being inferred
       from the postal sector (first two digits) of blocks matched in pass 0/1.
       Accepted only when exactly one unused property row qualifies.
    """
    prop_keys = set(prop["join_key"])
    keys = hdb["join_key"].copy()
    status = pd.Series("unmatched", index=hdb.index)

    # Pass 0: direct key match; label whether the abbreviation step was needed.
    raw_key = hdb["blk_norm"] + "_" + hdb["decoded_street"].str.strip().str.upper()
    hit = keys.isin(prop_keys)
    status[hit] = (raw_key[hit] == keys[hit]).map({True: "exact", False: "normalized"})

    # Pass 1: alternative spellings, never reusing a property row already taken.
    used = set(keys[hit])
    for i in hdb.index[~hit & hdb["decoded_street"].notna()]:
        street = hdb.at[i, "norm_street"]
        for candidate in (
            raw_key[i],
            hdb.at[i, "blk_norm"] + "_" + " ".join(EXTRA_ABBR.get(w, w) for w in street.split()),
        ):
            if candidate in prop_keys and candidate not in used:
                keys[i], status[i] = candidate, "alt_spelling"
                used.add(candidate)
                break

    # Pass 2: undecoded road codes, matched by block number within the postal sector's town(s).
    town_by_key = prop.set_index("join_key")["bldg_contract_town"]
    matched = status != "unmatched"
    sector_towns = (
        pd.DataFrame({
            "sector": hdb.loc[matched, "POSTAL_COD"].str[:2],
            "town": keys[matched].map(town_by_key),
        })
        .groupby("sector")["town"].agg(set)
    )
    unused = prop[~prop["join_key"].isin(used)]
    for i in hdb.index[(status == "unmatched") & hdb["decoded_street"].isna()]:
        towns = sector_towns.get(str(hdb.at[i, "POSTAL_COD"])[:2], set())
        candidates = unused[
            (unused["blk_norm"] == hdb.at[i, "blk_norm"]) & unused["bldg_contract_town"].isin(towns)
        ]
        if len(candidates) == 1:
            keys[i], status[i] = candidates["join_key"].iloc[0], "postal_sector"
            unused = unused.drop(candidates.index)

    return keys, status


# 1. Resolve every block's key and record how it was matched.
hdb["join_key"], hdb["match_status"] = resolve_keys(hdb, prop)
counts = hdb["match_status"].value_counts()
print(counts.to_string())

# 2. The headline rates: first pass alone, then after the fallbacks.
n = len(hdb)
first_pass = int(counts.get("exact", 0) + counts.get("normalized", 0))
matched = int((hdb["match_status"] != "unmatched").sum())
print(f"\nFirst pass (exact + normalized): {first_pass:,}/{n:,} = {first_pass / n:.1%}; "
      f"left unmatched after the first pass: {n - first_pass}")
print(f"After the fallback passes: {matched:,}/{n:,} = {matched / n:.1%}; still unmatched: {n - matched}")

match_status
normalized       11924
exact             1356
unmatched          120
postal_sector       19
alt_spelling        17

First pass (exact + normalized): 13,280/13,436 = 98.8%; left unmatched after the first pass: 156
After the fallback passes: 13,316/13,436 = 99.1%; still unmatched: 120


## 5. Join the attributes and project to metres

Left join, so unmatched blocks stay in the layer with null attributes; `footprint_m2` is computed in
the projected CRS and is needed for the imputation in the next section.

In [6]:
# 1. Left join keeps all 13,436 polygons; unmatched ones get null attributes.
joined = hdb.merge(prop[["join_key"] + PROPERTY_COLS], on="join_key", how="left")

# 2. Project to SVY21 so footprint areas are in square metres.
joined = joined.to_crs(CRS_OUT)
joined["footprint_m2"] = joined.geometry.area
print(f"Joined table: {len(joined):,} blocks, {joined['total_dwelling_units'].notna().sum():,} with attributes; "
      f"median footprint {joined['footprint_m2'].median():.0f} m2")

Joined table: 13,436 blocks, 13,316 with attributes; median footprint 1090 m2


## 6. Manual review of four blocks and imputation

Four unmatched blocks are full-size buildings on streets that do exist in Property Information,
but their block number does not (checked by postal code on OneMap and Google Maps, HDB resale
records and news reports; evidence in section 9). One is a community club and stays excluded;
three are residential and are included, with attributes imputed from the nearest matched
residential blocks on the same street: dwelling units = own footprint × the neighbours' median units
per m²; flat-type mix, floors, completion year and use flags from the same neighbours. Two of them
are SERS blocks reused for temporary housing and carry `review_flag = verify_occupancy`.

In [7]:
# Manual decisions for unmatched blocks checked by hand (OneMap, Google Maps, news), keyed by join_key:
# (decision, review flag, note).
MANUAL_REVIEW = {
    "87_JLN SATU": ("exclude", None, "Mountbatten Community Club, not residential"),
    "63_C'WEALTH DR": ("include", None, "Residential HDB block"),
    "82_MACPHERSON LANE": ("include", "verify_occupancy",
                           "SERS block (2018) repurposed for PPHS temporary housing"),
    "40_TANGLIN HALT RD": ("include", "verify_occupancy",
                           "SERS block with PPHS redevelopment plans"),
}


def impute_manual_includes(joined):
    """Fill attributes of MANUAL_REVIEW "include" blocks from same-street neighbours.

    Dwelling units = own footprint x median units per m2 of the IMPUTE_NEIGHBOURS
    nearest matched residential blocks on the same street; the flat-type mix, floors,
    year and flags are taken from those neighbours. Sets match_status to "imputed".
    """
    keys = [k for k, (decision, _, _) in MANUAL_REVIEW.items() if decision == "include"]
    targets = joined.index[joined["join_key"].isin(keys) & joined["match_status"].eq("unmatched")]
    for i in targets:
        pool = joined[joined["norm_street"].eq(joined.at[i, "norm_street"])
                      & joined["residential"].eq("Y")]
        nearest = pool.loc[
            pool.geometry.centroid.distance(joined.geometry[i].centroid).nsmallest(IMPUTE_NEIGHBOURS).index
        ]
        units = round(joined.at[i, "footprint_m2"]
                      * (nearest["total_dwelling_units"] / nearest["footprint_m2"]).median())
        mix = nearest[UNIT_COLS].sum() / nearest[UNIT_COLS].sum().sum()
        joined.loc[i, UNIT_COLS] = (mix * units).round().to_numpy()
        joined.at[i, "total_dwelling_units"] = joined.loc[i, UNIT_COLS].sum()
        joined.at[i, "max_floor_lvl"] = nearest["max_floor_lvl"].median()
        joined.at[i, "year_completed"] = nearest["year_completed"].median()
        for col in ["bldg_contract_town", "residential", "commercial", "market_hawker",
                    "miscellaneous", "multistorey_carpark", "precinct_pavilion"]:
            joined.at[i, col] = nearest[col].mode().iloc[0]
        joined.at[i, "match_status"] = "imputed"
        print(f"  imputed {joined.at[i, 'join_key']}: {int(joined.at[i, 'total_dwelling_units'])} units "
              f"from {', '.join(nearest['BLK_NO'])}")
    return joined


# 1. Impute the three included blocks (the excluded one simply keeps null attributes).
print("Imputed blocks:")
joined = impute_manual_includes(joined)

Imputed blocks:
  imputed 63_C'WEALTH DR: 116 units from 62, 64, 60
  imputed 82_MACPHERSON LANE: 113 units from 83, 81
  imputed 40_TANGLIN HALT RD: 118 units from 41, 42, 38


## 7. Derived fields

`gfa_m2` (footprint × floors) is the team's agreed weight for allocating population when flat
counts are not usable; `is_residential` drops carparks, markets and pavilions from the demand side.
Unmatched blocks have `residential` null, so they are not residential and carry no demand.

In [8]:
# 1. Gross floor area proxy and the residential flag.
joined["gfa_m2"] = joined["footprint_m2"] * joined["max_floor_lvl"]
joined["is_residential"] = joined["residential"].eq("Y")

# 2. Flags from the manual review travel with the layer so downstream users see them.
joined["review_flag"] = joined["join_key"].map(
    {k: flag for k, (_, flag, _) in MANUAL_REVIEW.items() if flag}
)

n = len(joined)
n_matched = int((joined["match_status"] != "unmatched").sum())
print(f"Nationwide HDB join: {n_matched:,}/{n:,} matched ({n_matched / n:.1%})")
print(joined["match_status"].value_counts().to_string())
print(f"Residential blocks: {int(joined['is_residential'].sum()):,}, "
      f"dwelling units: {int(joined['total_dwelling_units'].sum()):,}")
print(f"Blocks flagged for review: {int(joined['review_flag'].notna().sum())}")

Nationwide HDB join: 13,319/13,436 matched (99.1%)
match_status
normalized       11924
exact             1356
unmatched          117
postal_sector       19
alt_spelling        17
imputed              3
Residential blocks: 10,769, dwelling units: 1,172,065
Blocks flagged for review: 2


## 8. Write the layer and the review file

The review file lists every block still without its own Property Information row: 117 excluded
(`unmatched`) and 3 included with imputed attributes (`imputed`), with the reason and the manual
decision. The 36 blocks recovered by the fallback passes are not in it; they carry their
`match_status` in the layer.

In [9]:
# 1. All blocks, matched or not, go to the GeoPackage; downstream filters on is_residential.
joined.to_file(OUT_GPKG, layer=OUT_LAYER, driver="GPKG")
print(f"Wrote layer '{OUT_LAYER}' ({n:,} features, {CRS_OUT}) to {OUT_GPKG.relative_to(ROOT)}")

# 2. Review file: why each block has no Property Information row and what was decided.
unmatched = joined[joined["match_status"].isin(["unmatched", "imputed"])].copy()
unmatched["reason"] = unmatched["decoded_street"].isna().map(
    {True: "undecoded_road_code", False: "no_match_in_property_info"}
)
# Unmatched blocks are excluded from demand (is_residential is False for them).
unmatched["resolution"] = unmatched["match_status"].map(
    {"imputed": "included_imputed", "unmatched": "excluded_not_in_property_info"}
)
unmatched["review_note"] = unmatched["join_key"].map(
    {k: note for k, (_, _, note) in MANUAL_REVIEW.items()}
)
unmatched[["BLK_NO", "ST_COD", "POSTAL_COD", "decoded_street", "norm_street",
           "footprint_m2", "total_dwelling_units", "reason", "resolution",
           "review_flag", "review_note"]].to_csv(REVIEW_CSV, index=False)
print(f"Wrote {len(unmatched)} unmatched/imputed rows to {REVIEW_CSV.relative_to(ROOT)}")
print(unmatched["resolution"].value_counts().to_string())
print("\nStreets of the excluded blocks:")
print(unmatched.loc[unmatched["match_status"] == "unmatched", "norm_street"].value_counts().to_string())

Wrote layer 'hdb_blocks' (13,436 features, EPSG:3414) to output/hdb_blocks.gpkg
Wrote 120 unmatched/imputed rows to output/hdb_join_review.csv
resolution
excluded_not_in_property_info    117
included_imputed                   3

Streets of the excluded blocks:
norm_street
PASIR RIS HTS    55
PASIR RIS TER    40
PASIR RIS AVE    15
PASIR RIS RD      6
JLN SATU          1


## 9. How the 156 unmatched blocks were handled

*Text of `output/hdb_unresolved_blocks_handling.md` (Minh, 2026-10-22), kept with the code that implements it.*

`scripts/join_hdb_nationwide.py` joins the HDB Existing Building polygons (2026-06-21) to
HDB Property Information (2026-10-08) on block number + street name. The first pass left
**156 of 13,436 blocks** without attributes. Every one of them is listed, with its
decision, in [`hdb_join_review.csv`](../output/hdb_join_review.csv) (resolved blocks carry their
`match_status` in `hdb_blocks.gpkg`).

| Outcome | Blocks | `match_status` |
|---|---:|---|
| Matched by a fallback pass | 36 | `postal_sector`, `alt_spelling` |
| Included after manual review, units imputed | 3 | `imputed` |
| Excluded | 117 | `unmatched` |
| **Total** | **156** | |

### 9.1 Matched by a fallback pass (36)

**Road codes newer than the 2019 LTA lookup (19).** `ST_COD` `HOS21Q` and `HOS22Q` are
missing from `LTARoadNameCode_2019.xlsx`, so no street name could be decoded. These blocks
are matched by block number among property rows in the same contract town, where the town
is inferred from the postal sector (first two digits) of already-matched blocks; a match
is accepted only when exactly one unused row qualifies.
`HOS21Q` → Hougang St 12 (9 blocks), `HOS22Q` → Hougang St 13 (10 blocks), all completed
2025–2026.

**Street-name spelling variants (17).** The abbreviation table shortens words that HDB keeps
in full in some names, or misses abbreviations HDB uses only sometimes. The fallback tries
the unabbreviated LTA name and an extra abbreviation set (`MARKET → MKT`, `INDUSTRIAL → IND`):
Buangkok Green (10), Taman Ho Swee (5), New Market Rd → New Mkt Rd (1),
Woodlands Industrial Park E3 → Woodlands Ind Pk E3 (1).

25 of the 36 are residential; they now hold about 467 children in
`hdb_blocks_children_final.gpkg`.

### 9.2 Manual review of single blocks (4)

Four unmatched blocks are full-size buildings (900–2,400 m², polygons updated 2024–25) on
streets that do exist in Property Information, but their block number does not. Each was
checked by postal code on OneMap and Google Maps, against HDB resale records
(data.gov.sg, 2017 onwards) and news reports.

| Block | Postal | Evidence | Decision |
|---|---|---|---|
| 87 Jalan Satu | 390087 | OneMap: Mountbatten Community Club | Exclude |
| 63 Commonwealth Dr | 140063 | Residential block on Google Maps | Include |
| 82 MacPherson Lane | 360082 | 3-room flats (1968 lease), last resale Oct 2017; selected for SERS in 2018 ([Straits Times](https://www.straitstimes.com/singapore/three-blocks-in-macpherson-lane-get-nod-for-sers)), since repurposed for PPHS temporary housing | Include, flag |
| 40 Tanglin Halt Rd | 143040 | Residential block on Google Maps; SERS block with PPHS redevelopment plans | Include, flag |

The decisions are coded in `MANUAL_REVIEW` in `join_hdb_nationwide.py`. Flagged blocks
carry `review_flag = verify_occupancy` in both `hdb_blocks.gpkg` and
`hdb_blocks_children_final.gpkg`; occupancy was not independently verified.

**Imputed attributes.** With no Property Information row, the 3 included blocks take their
attributes from the nearest residential blocks on the same street: dwelling units = own
footprint × the neighbours' median units per m²; flat-type mix, floors, completion year and
use flags from the same neighbours.

| Block | Neighbours used | Est. units | Children 0–9 |
|---|---|---:|---:|
| 63 Commonwealth Dr | 60, 62, 64 (1963, 116–118 units each) | 116 | 13.7 |
| 82 MacPherson Lane | 81, 83 (1968, 130 and 93 units) | 113 | 11.3 |
| 40 Tanglin Halt Rd | 38, 41, 42 (1962, 116–118 units each) | 118 | 13.9 |

The neighbours for 63 and 40 are near-identical blocks; the estimate for 82 is rougher.

### 9.3 Rows that remain excluded (117)

- **116 Pasir Ris structures**: Pasir Ris Heights (55), Terrace (40), Avenue (15),
  Road (6). None of these streets appears in Property Information 2026, the footprints are
  small (median 96 m² against roughly 600–1,500 m² for a flat block) and most polygons were
  last updated in 2013. They are not treated as flat blocks. Verified that they are landed properties through google search.
- **87 Jalan Satu**: Mountbatten Community Club (see §2).

Excluded blocks have null attributes and `is_residential = False`, so they carry no demand.

### 9.4 Effect on child population

Subzone child totals come from SingStat and are always fully assigned to the residential
blocks in that subzone, so no child is lost through an excluded block; an exclusion can
only shift children to other blocks in the same subzone. After this review all 259,910
HDB children aged 0–9 are assigned across 10,769 blocks, with no nulls. The 3 imputed
blocks hold about 39 of them.

### Regenerate

Run this notebook, then `1_04_assign_child_population.ipynb`.

## 10. Files written

In [10]:
info = pyogrio.read_info(OUT_GPKG, layer=OUT_LAYER)
print(f"{OUT_GPKG.relative_to(ROOT)} layer {OUT_LAYER}: {info['features']:,} blocks, {info['crs']}")
print(f"{REVIEW_CSV.relative_to(ROOT)}: {len(pd.read_csv(REVIEW_CSV)):,} rows")

output/hdb_blocks.gpkg layer hdb_blocks: 13,436 blocks, EPSG:3414
output/hdb_join_review.csv: 120 rows
